# Databricks + PySpark Lakehouse Demo (batch)
**Objetivo:** demostrar carga y procesamiento batch en un data lake/lakehouse sobre Databricks usando datasets reales.

## Qué cubre este notebook
1. Cargar `.csv`
2. Cargar `.json`
3. Cargar `.parquet`
4. Convertir `.csv` a `.parquet`
5. Convertir a **Delta Lake** y probar funcionalidades
6. Operaciones sobre DataFrames: agregar/eliminar columnas, joins, filtros, agregaciones
7. EDA con Spark
8. SQL sobre Spark
9. Regresión lineal supervisada
10. K-Means
11. Ejemplos batch más avanzados: ventanas, deduplicación, medallion pattern, time travel

## Datasets reales usados
- **NYC Open Data – 311 Service Requests** en CSV y JSON
- **NYC TLC Yellow Taxi Trips** en Parquet

> Diseñado para ejecutarse en un notebook de **Databricks** con runtime que ya incluya Apache Spark y Delta Lake.

> EN LA VERSION FREE TIER NO EJECUTA MODELOS Spark ML en computo Serverless.

In [0]:

# Parámetro base para que todo quede organizado en DBFS
base_path = "/Volumes/mycatalog/default/myvolume/demo/"
raw_path = f"{base_path}/raw"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
ml_path = f"{base_path}/ml"

for p in [base_path, raw_path, bronze_path, silver_path, gold_path, ml_path]:
    dbutils.fs.mkdirs(p)

display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
dbfs:/Volumes/mycatalog/default/myvolume/demo/bronze/,bronze/,0,1775881452542
dbfs:/Volumes/mycatalog/default/myvolume/demo/gold/,gold/,0,1775881452542
dbfs:/Volumes/mycatalog/default/myvolume/demo/ml/,ml/,0,1775881452542
dbfs:/Volumes/mycatalog/default/myvolume/demo/raw/,raw/,0,1775881452542
dbfs:/Volumes/mycatalog/default/myvolume/demo/silver/,silver/,0,1775881452542


## 1) Descarga de datasets reales hacia DBFS

En Databricks, una forma práctica de trabajar con archivos remotos es descargarlos primero a `/dbfs/...`
y luego leerlos con Spark desde `dbfs:/...`.

In [0]:

import os
import urllib.request

local_base = "/Volumes/mycatalog/default/myvolume/demo/raw"
os.makedirs(local_base, exist_ok=True)

files = {
    # CSV real desde NYC Open Data (311)
    "311_sample.csv": "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=50000",

    # JSON real desde NYC Open Data (311)
    "311_sample.json": "https://data.cityofnewyork.us/resource/erm2-nwe9.json?$limit=20000",

    # Parquet real desde NYC TLC
    "yellow_tripdata_2025-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
}

for filename, url in files.items():
    target = os.path.join(local_base, filename)
    if not os.path.exists(target):
        print(f"Descargando {filename} ...")
        urllib.request.urlretrieve(url, target)
    else:
        print(f"Ya existe: {filename}")

print("Archivos descargados en:", local_base)

Ya existe: 311_sample.csv
Ya existe: 311_sample.json
Ya existe: yellow_tripdata_2025-01.parquet
Archivos descargados en: /Volumes/mycatalog/default/myvolume/demo/raw


In [0]:

display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
dbfs:/Volumes/mycatalog/default/myvolume/demo/raw/311_sample.csv,311_sample.csv,42978813,1775877037000
dbfs:/Volumes/mycatalog/default/myvolume/demo/raw/311_sample.json,311_sample.json,29383577,1775877114000
dbfs:/Volumes/mycatalog/default/myvolume/demo/raw/yellow_tripdata_2025-01.parquet,yellow_tripdata_2025-01.parquet,59158238,1775877121000


## 2) Cargar archivos CSV

In [0]:

csv_path = f"{raw_path}/311_sample.csv"

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

print("Schema CSV:")
df_csv.printSchema()
display(df_csv.limit(10))

Schema CSV:
root
 |-- unique_key: integer (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- incident_zip: integer (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- intersection_street_1: string (nullable = true)
 |-- intersection_street_2: string (nullable = true)
 |-- address_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- landmark: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- due_date: timestamp (nullable = true)
 |-- reso

unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
68599339,2026-04-09T02:05:14.000Z,null,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,null,Street/Sidewalk,10473,1674 SEWARD AVENUE,SEWARD AVENUE,METCALF AVENUE EXTENSION,CROES PLACE,METCALF AVENUE EXTENSION,CROES PLACE,ADDRESS,BRONX,SEWARD AVENUE,null,In Progress,null,null,null,09 BRONX,18,Precinct 43,2035510001,BRONX,1020279,237416,MOBILE,Unspecified,BRONX,null,null,null,null,null,null,null,40.81825427441717,-73.86983220090866,POINT (-73.869832200909 40.818254274417)
68600674,2026-04-09T02:04:42.000Z,null,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,null,Residential Building/House,11214,1869 83 STREET,83 STREET,18 AVENUE,19 AVENUE,18 AVENUE,19 AVENUE,ADDRESS,BROOKLYN,83 STREET,null,In Progress,null,null,null,11 BROOKLYN,38,Precinct 62,3063150039,BROOKLYN,984421,160824,PHONE,Unspecified,BROOKLYN,null,null,null,null,null,null,null,40.60810108364675,-73.99938414563323,POINT (-73.999384145633 40.608101083647)
68599302,2026-04-09T02:04:07.000Z,null,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,null,Street/Sidewalk,11102,18-27 25 ROAD,25 ROAD,18 STREET,21 STREET,18 STREET,21 STREET,ADDRESS,ASTORIA,25 ROAD,null,In Progress,null,null,null,01 QUEENS,22,Precinct 114,4008880008,QUEENS,1004798,221531,ONLINE,Unspecified,QUEENS,null,null,null,null,null,null,null,40.77470385504901,-73.92581148357691,POINT (-73.925811483577 40.774703855049)
68597918,2026-04-09T02:04:05.000Z,null,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,null,Street/Sidewalk,11365,67-10 190 LANE,190 LANE,192 STREET,BEND,192 STREET,BEND,ADDRESS,FRESH MEADOWS,190 LANE,null,In Progress,null,null,null,08 QUEENS,23,Precinct 107,null,QUEENS,1044772,207855,MOBILE,Unspecified,QUEENS,null,null,null,null,null,null,null,40.73698352494025,-73.78160893379135,POINT (-73.781608933791 40.73698352494)
68605894,2026-04-09T02:02:29.000Z,null,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,null,Residential Building/House,11368,35-38 JUNCTION BOULEVARD,JUNCTION BOULEVARD,35 AVENUE,37 AVENUE,35 AVENUE,37 AVENUE,ADDRESS,CORONA,JUNCTION BOULEVARD,null,In Progress,null,null,null,03 QUEENS,21,Precinct 115,4014697501,QUEENS,1019814,213646,ONLINE,Unspecified,QUEENS,null,null,null,null,null,null,null,40.75301392618723,-73.87163802142506,POINT (-73.871638021425 40.753013926187)
68600636,2026-04-09T02:02:25.000Z,null,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,null,Street/Sidewalk,11365,67-44 190 LANE,190 LANE,BEND,69 AVENUE,BEND,69 AVENUE,ADDRESS,FRESH MEADOWS,190 LANE,null,In Progress,null,null,null,08 QUEENS,23,Precinct 107,4071170003,QUEENS,1044540,207698,MOBILE,Unspecified,QUEENS,null,null,null,null,null,null,null,40.73655418461277,-73.78244749948752,POINT (-73.782447499488 40.736554184613)
68603313,2026-04-09T02:02:08.000Z,null,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,null,Residential Building/House,11365,49-80 175 PLACE,175 PLACE,PECK AVENUE,50 AVENUE,PECK AVENUE,50 AVENUE,ADDRESS,FRESH MEADOWS,175 PLACE,null,In Progress,null,null,null,11 QUEENS,20,Precinct 111,4055910040,QUEENS,1041451,210714,MOBILE,Unspecified,QUEENS,null,null,null,null,null,null,null,40.74485284901543,-73.79356826355485,POINT (-73.793568263555 40.744852849015)
68601940,2026-04-09T02:00:30.000

## 3) Cargar archivos JSON

In [0]:
json_path = f"{raw_path}/311_sample.json"

df_json = (
    spark.read
    .option("multiline", True)
    .json(json_path)
)

print("Schema JSON:")
df_json.printSchema()
display(df_json.limit(10))

Schema JSON:
root
 |-- :@computed_region_92fq_4b7q: string (nullable = true)
 |-- :@computed_region_f5dn_yrer: string (nullable = true)
 |-- :@computed_region_sbqj_enih: string (nullable = true)
 |-- :@computed_region_yeji_bk3q: string (nullable = true)
 |-- address_type: string (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- bridge_highway_direction: string (nullable = true)
 |-- bridge_highway_name: string (nullable = true)
 |-- bridge_highway_segment: string (nullable = true)
 |-- city: string (nullable = true)
 |-- closed_date: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- descriptor: string (nulla

:@computed_region_92fq_4b7q,:@computed_region_f5dn_yrer,:@computed_region_sbqj_enih,:@computed_region_yeji_bk3q,address_type,agency,agency_name,bbl,borough,bridge_highway_direction,bridge_highway_name,bridge_highway_segment,city,closed_date,community_board,complaint_type,council_district,created_date,cross_street_1,cross_street_2,descriptor,descriptor_2,due_date,facility_type,incident_address,incident_zip,intersection_street_1,intersection_street_2,landmark,latitude,location,location_type,longitude,open_data_channel_type,park_borough,park_facility_name,police_precinct,resolution_action_updated_date,resolution_description,road_ramp,status,street_name,taxi_company_borough,taxi_pick_up_location,unique_key,vehicle_type,x_coordinate_state_plane,y_coordinate_state_plane
31,58,26,5,ADDRESS,NYPD,New York City Police Department,2035510001,BRONX,null,null,null,BRONX,null,09 BRONX,Noise - Street/Sidewalk,18,2026-04-09T02:05:14.000,METCALF AVENUE EXTENSION,CROES PLACE,Loud Talking,null,null,null,1674 SEWARD AVENUE,10473,METCALF AVENUE EXTENSION,CROES PLACE,SEWARD AVENUE,40.81825427441717,"List(List(-73.869832200909, 40.818254274417), Point)",Street/Sidewalk,-73.86983220090866,MOBILE,BRONX,Unspecified,Precinct 43,null,null,null,In Progress,SEWARD AVENUE,null,null,68599339,null,1020279,237416
45,1,37,2,ADDRESS,NYPD,New York City Police Department,3063150039,BROOKLYN,null,null,null,BROOKLYN,null,11 BROOKLYN,Noise - Residential,38,2026-04-09T02:04:42.000,18 AVENUE,19 AVENUE,Banging/Pounding,null,null,null,1869 83 STREET,11214,18 AVENUE,19 AVENUE,83 STREET,40.60810108364675,"List(List(-73.999384145633, 40.608101083647), Point)",Residential Building/House,-73.99938414563323,PHONE,BROOKLYN,Unspecified,Precinct 62,null,null,null,In Progress,83 STREET,null,null,68600674,null,984421,160824
4,39,72,3,ADDRESS,NYPD,New York City Police Department,4008880008,QUEENS,null,null,null,ASTORIA,null,01 QUEENS,Illegal Parking,22,2026-04-09T02:04:07.000,18 STREET,21 STREET,Blocked Hydrant,null,null,null,18-27 25 ROAD,11102,18 STREET,21 STREET,25 ROAD,40.77470385504901,"List(List(-73.925811483577, 40.774703855049), Point)",Street/Sidewalk,-73.92581148357691,ONLINE,QUEENS,Unspecified,Precinct 114,null,null,null,In Progress,25 ROAD,null,null,68599302,null,1004798,221531
16,25,65,3,ADDRESS,NYPD,New York City Police Department,null,QUEENS,null,null,null,FRESH MEADOWS,null,08 QUEENS,Illegal Parking,23,2026-04-09T02:04:05.000,192 STREET,BEND,Blocked Hydrant,null,null,null,67-10 190 LANE,11365,192 STREET,BEND,190 LANE,40.73698352494025,"List(List(-73.781608933791, 40.73698352494), Point)",Street/Sidewalk,-73.78160893379135,MOBILE,QUEENS,Unspecified,Precinct 107,null,null,null,In Progress,190 LANE,null,null,68597918,null,1044772,207855
21,65,73,3,ADDRESS,NYPD,New York City Police Department,4014697501,QUEENS,null,null,null,CORONA,null,03 QUEENS,Noise - Residential,21,2026-04-09T02:02:29.000,35 AVENUE,37 AVENUE,Banging/Pounding,null,null,null,35-38 JUNCTION BOULEVARD,11368,35 AVENUE,37 AVENUE,JUNCTION BOULEVARD,40.75301392618723,"List(List(-73.871638021425, 40.753013926187), Point)",Residential Building/House,-73.87163802142506,ONLINE,QUEENS,Unspecified,Precinct 115,null,null,null,In Progress,JUNCTION BOULEVARD,null,null,68605894,null,1019814,213646
16,25,65,3,ADDRESS,NYPD,New York City Police Department,4071170003,QUEENS,null,null,null,FRESH MEADOWS,null,08 QUEENS,Illegal Parking,23,2026-04-09T02:02:25.000,BEND,69 AVENUE,Blocked Hydrant,null,null,null,67-44 190 LANE,11365,BEND,69 AVENUE,190 LANE,40.73655418461277,"List(List(-73.782447499488, 40.736554184613), Point)",Street/Sidewalk,-73.78244749948752,MOBILE,QUEENS,Unspecified,Precinct 107,null,null,null,In Progress,190 LANE,null,null,68600636,null,1044540,207698
3,26,69,3,ADDRESS,NYPD,New York City Police Department,4055910040,QUEENS,null,null,null,FRESH MEADOWS,null,11 QUEENS,Noise - Residential,20,2026-04-09T02:02:08.000,PECK AVENUE,50 AVENUE,Loud Music/Party,null,null,null,49-80 175 PLACE,11365,PECK AVENUE,50 AVENUE

## 4) Cargar archivos Parquet

In [0]:

parquet_path = f"{raw_path}/yellow_tripdata_2025-01.parquet"

df_taxi = spark.read.parquet(parquet_path)

print("Schema Parquet:")
df_taxi.printSchema()
display(df_taxi.limit(10))

Schema Parquet:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0


## 5) Convertir CSV a Parquet

Guardamos una copia del dataset 311 en Parquet para mostrar conversión de formato.

In [0]:

csv_to_parquet_path = f"{bronze_path}/311_parquet"

(
    df_csv.write
    .mode("overwrite")
    .parquet(csv_to_parquet_path)
)

df_311_parquet = spark.read.parquet(csv_to_parquet_path)

print("Filas en CSV original:", df_csv.count())
print("Filas en Parquet convertido:", df_311_parquet.count())
display(df_311_parquet.limit(10))

Filas en CSV original: 50000
Filas en Parquet convertido: 50000


unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
68592790,2026-04-07T17:01:49.000Z,null,DSNY,Department of Sanitation,Graffiti,Graffiti,null,Mixed Use,10002,155 ALLEN STREET,ALLEN STREET,null,null,null,null,ADDRESS,NEW YORK,null,N/A,Open,2026-05-07T17:01:49.000Z,The graffiti on this property has been scheduled to be removed by the City.,2026-04-07T17:01:49.000Z,03 MANHATTAN,1,Precinct 5,1004160028,MANHATTAN,987152,201910,UNKNOWN,Unspecified,MANHATTAN,null,null,null,null,null,null,null,40.7208725527404,-73.98953079850072,POINT (-73.989530798501 40.72087255274)
68589422,2026-04-07T17:01:46.000Z,null,DOT,Department of Transportation,Street Condition,Pothole,null,null,11420,null,null,null,null,115 STREET,150 AVENUE,INTERSECTION,QUEENS,null,N/A,Open,null,The Department of Transportation referred this complaint to the appropriate Maintenance Unit for repair.,2026-04-07T17:01:46.000Z,10 QUEENS,31,Precinct 106,null,QUEENS,1032605,182540,UNKNOWN,Unspecified,QUEENS,null,null,null,null,null,null,null,40.6675748180031,-73.82569467462247,POINT (-73.825694674622 40.667574818003)
68587286,2026-04-07T17:01:43.000Z,2026-04-07T19:55:18.000Z,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,null,Street/Sidewalk,10460,626 BAKER AVENUE,BAKER AVENUE,GARFIELD STREET,UNIONPORT ROAD,GARFIELD STREET,UNIONPORT ROAD,ADDRESS,BRONX,BAKER AVENUE,null,Closed,null,"The New York City Police Department responded to the complaint and their investigation determined that no criminal violation existed. The condition was corrected without the need to issue a summons or effect an arrest. If the problem persists, please contact 311 to create another complaint. If possible, provide contact information so responding officers may reach out to you for more details. If necessary, your complaint may be referred to your local precinct's special operations units (Quality of Life, etc.). Thank you for your attention to this matter. We count on New Yorkers like yourself to maintain a safe City, so please let us know if you see other conditions that require our attention.",2026-04-07T19:55:21.000Z,11 BRONX,13,Precinct 49,2040250012,BRONX,1021249,245985,ONLINE,Unspecified,BRONX,null,null,null,null,null,null,null,40.84176968733819,-73.86628047117955,POINT (-73.86628047118 40.841769687338)
68593415,2026-04-07T17:01:39.000Z,2026-04-08T18:48:57.000Z,HPD,Department of Housing Preservation and Development,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT,RESIDENTIAL BUILDING,10032,900 RIVERSIDE DRIVE,RIVERSIDE DRIVE,null,null,null,null,ADDRESS,NEW YORK,null,null,Closed,null,HPD conducted an inspection of this complaint. The conditions observed by the inspector did not violate the housing laws enforced by HPD. The complaint has been closed.,2026-04-08T00:00:00.000Z,12 MANHATTAN,7,Precinct 33,1021360167,MANHATTAN,999049,244634,ONLINE,Unspecified,MANHATTAN,null,null,null,null,null,null,null,40.83812669619484,-73.94651730320311,POINT (-73.946517303203 40.838126696195)
68596108,2026-04-07T17:01:30.000Z,2026-04-08T14:37:43.000Z,HPD,Department of Housing Preservation and Development,HEAT/HOT WATER,APARTMENT ONLY,NO HEAT,RESIDENTIAL BUILDING,10039,2569 7 AVENUE,7 AVENUE,null,null,null,null,ADDRESS,NEW YORK,null,null,Closed,null,"This complaint is a duplicate of a building-wide condition already reported by another tenant. The original complaint is still open, and HPD ma

## 6) Operaciones de DataFrame
- seleccionar columnas
- crear columnas derivadas
- eliminar columnas
- filtrar
- agregar
- unir DataFrames

In [0]:
from pyspark.sql import functions as F

taxi_enriched = (
    df_taxi
    .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("speed_mph",
                F.when(F.col("trip_duration_min") > 0,
                       F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)))
)

display(taxi_enriched.select(
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "trip_distance", "fare_amount", "tip_amount",
    "trip_duration_min", "pickup_hour", "speed_mph"
).limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,tip_amount,trip_duration_min,pickup_hour,speed_mph
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1.6,10.0,3.0,8.35,0,11.497005988023954
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,0.5,5.1,2.02,2.55,0,11.764705882352942
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,0.6,5.1,2.0,1.95,0,18.46153846153846
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,0.52,7.2,0.0,5.566666666666666,0,5.604790419161676
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,0.66,5.8,0.0,3.533333333333333,0,11.207547169811322
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2.63,19.1,0.0,20.033333333333335,0,7.876871880199666
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0.4,4.4,2.35,1.4666666666666666,0,16.363636363636367
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,1.6,12.1,2.0,12.4,0,7.741935483870968
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,2.8,19.1,3.0,19.666666666666668,0,8.54237288135593
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1.71,11.4,0.0,9.566666666666666,0,10.724738675958188


In [0]:

# Ejemplo de drop de columnas
taxi_reduced = taxi_enriched.drop("store_and_fwd_flag", "RatecodeID")
display(taxi_reduced.limit(5))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,speed_mph
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,8.35,2025-01-01,0,11.497005988023954
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.55,2025-01-01,0,11.764705882352942
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1.95,2025-01-01,0,18.46153846153846
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,5.566666666666666,2025-01-01,0,5.604790419161676
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,3.533333333333333,2025-01-01,0,11.207547169811322


In [0]:

# Filtrado de calidad básica
taxi_clean = (
    taxi_enriched
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_duration_min") > 0)
    .filter(F.col("trip_duration_min") < 240)
    .filter(F.col("passenger_count").isNotNull())
)

print("Filas limpias:", taxi_clean.count())
display(taxi_clean.limit(10))

Filas limpias: 2838026


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,speed_mph
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,8.35,2025-01-01,0,11.497005988023954
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.55,2025-01-01,0,11.764705882352942
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1.95,2025-01-01,0,18.46153846153846
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,5.566666666666666,2025-01-01,0,5.604790419161676
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,3.533333333333333,2025-01-01,0,11.207547169811322
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0,20.033333333333335,2025-01-01,0,7.876871880199666
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,1.4666666666666666,2025-01-01,0,16.363636363636367
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,12.4,2025-01-01,0,7.741935483870968
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,19.666666666666668,2025-01-01,0,8.54237288135593
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0,9.566666666666666,2025-01-01,0,10.724738675958188


In [0]:

# Agregaciones
daily_stats = (
    taxi_clean.groupBy("pickup_date")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip"),
        F.avg("trip_duration_min").alias("avg_duration_min")
    )
    .orderBy("pickup_date")
)

display(daily_stats)

pickup_date,num_trips,avg_trip_distance,avg_fare,avg_tip,avg_duration_min
2024-12-31,21,3.6576190476190478,19.123809523809523,4.05095238095238,16.805555555555557
2025-01-01,70813,3.9350335390394013,20.737836131783915,3.4767467837826564,14.591428127603617
2025-01-02,78112,3.7306040045063127,20.389869418271275,3.4930915864399896,16.203345410009746
2025-01-03,84313,3.420221436789108,19.082988032687773,3.394011362423342,15.532046461795266
2025-01-04,89912,3.273427796067251,18.51213531008134,3.3541500578342838,14.550140693122163
2025-01-05,72802,3.8862311474959075,20.058246614104174,3.6623388093734595,14.106422900469687
2025-01-06,73491,3.5732335932290966,19.11878175558926,3.585747506497428,14.81249790223744
2025-01-07,89355,3.0950494096581296,17.568067483632987,3.438113256113379,14.312134370395503
2025-01-08,96707,2.9470574001882133,17.076547923108095,3.4272352570135465,14.337217747078206
2025-01-09,100740,3.063757196744107,17.59964790549951,3.533607008139887,14.824906194163468


In [0]:

# Join sencillo entre agregados por vendor y una dimensión pequeña creada en el notebook
vendor_dim = spark.createDataFrame(
    [
        (1, "Creative Mobile Technologies"),
        (2, "VeriFone / Curb")
    ],
    ["VendorID", "vendor_name"]
)

vendor_stats = (
    taxi_clean.groupBy("VendorID")
    .agg(
        F.count("*").alias("trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_distance").alias("avg_distance")
    )
)

vendor_stats_joined = vendor_stats.join(vendor_dim, on="VendorID", how="left")
display(vendor_stats_joined.orderBy("VendorID"))

VendorID,trips,avg_fare,avg_distance,vendor_name
1,652128,18.47476266929179,3.3074600385200235,Creative Mobile Technologies
2,2185898,18.155089002324935,3.199421976688685,VeriFone / Curb


## 7) EDA con Spark

In [0]:

# Conteo, tipos y nulos
from pyspark.sql import functions as F

eda_cols = ["passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "trip_duration_min"]

null_summary = taxi_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in eda_cols
])

display(null_summary)

passenger_count,trip_distance,fare_amount,tip_amount,total_amount,trip_duration_min
0,0,0,0,0,0


In [0]:

# Descriptivos
display(taxi_clean.select(eda_cols).describe())

summary,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,trip_duration_min
count,2838026,2838026,2838026,2838026,2838026,2838026
mean,1.2956611391157093,3.2242472056280356,18.228544336804163,3.4963313901972257,27.65273007014269,14.585966155115031
stddev,0.7484427120999027,43.33171106445342,512.7676312976525,3.8166342315560673,512.9321188694169,11.758719922581548
min,0,0.01,0.01,0.0,1.01,0.016666666666666666
max,9,44730.3,863372.12,400.0,863380.37,238.91666666666666


In [0]:

# Cuantiles aproximados
quantiles = taxi_clean.approxQuantile(
    ["trip_distance", "fare_amount", "tip_amount", "trip_duration_min"],
    [0.25, 0.5, 0.75, 0.95],
    0.01
)
quantiles

[[1.0, 1.61, 3.07, 11.4],
 [8.6, 12.8, 19.8, 59.7],
 [1.04, 2.9, 4.29, 11.82],
 [7.183333333333334, 11.283333333333333, 17.966666666666665, 39.0]]

In [0]:

# Distribución por hora
hourly_stats = (
    taxi_clean.groupBy("pickup_hour")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_distance").alias("avg_distance")
    )
    .orderBy("pickup_hour")
)

display(hourly_stats)

pickup_hour,num_trips,avg_fare,avg_distance
0,65529,19.15107524912648,3.8060513665705065
1,43971,17.15034158877453,3.24719519683426
2,29643,16.192121917484638,2.9981671895557156
3,19286,17.219838224618872,3.2898247433371064
4,12401,23.1546762357874,4.786359164583508
5,15466,27.034527996896223,6.033790249579713
6,35161,21.586039361792942,5.834024345155139
7,74825,18.532275041764553,3.520626929502195
8,106867,16.984445525747,2.9303010283810607
9,123335,17.11167268009929,3.231968540965631


In [0]:

# Top zonas origen por volumen
pickup_zone_stats = (
    taxi_clean.groupBy("PULocationID")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip")
    )
    .orderBy(F.desc("num_trips"))
)

display(pickup_zone_stats.limit(20))

PULocationID,num_trips,avg_fare,avg_tip
237,149378,12.000238589350772,2.6768826734860536
161,147984,14.991290342199344,3.2530854011248054
236,139057,12.500444781636645,2.732516809653359
132,134059,62.72446639166139,9.359567354671018
186,109227,15.370215789136342,3.1543192617211955
230,108379,17.437669382444803,3.4859548436507626
162,105910,14.640820885657806,3.2066565952225634
142,98534,13.383038849534415,2.942352385978526
138,85799,52.24933250970321,9.032221704215598
163,85328,15.204833231764434,3.2369996952935804


## 8) SQL sobre Spark

In [0]:

taxi_clean.createOrReplaceTempView("taxi_clean")
df_csv.createOrReplaceTempView("service_requests_csv")

In [0]:

# Consulta SQL 1: resumen por hora
hourly_sql = spark.sql("""
SELECT
  hour(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM taxi_clean
GROUP BY hour(tpep_pickup_datetime)
ORDER BY pickup_hour
""")
display(hourly_sql)

pickup_hour,trips,avg_fare,avg_distance
0,65529,19.15,3.81
1,43971,17.15,3.25
2,29643,16.19,3.0
3,19286,17.22,3.29
4,12401,23.15,4.79
5,15466,27.03,6.03
6,35161,21.59,5.83
7,74825,18.53,3.52
8,106867,16.98,2.93
9,123335,17.11,3.23


In [0]:

# Consulta SQL 2: top categorías 311
requests_sql = spark.sql("""
SELECT complaint_type, COUNT(*) AS total
FROM service_requests_csv
WHERE complaint_type IS NOT NULL
GROUP BY complaint_type
ORDER BY total DESC
LIMIT 20
""")
display(requests_sql)

complaint_type,total
Illegal Parking,8035
Noise - Residential,5689
HEAT/HOT WATER,4647
Street Condition,2674
Blocked Driveway,2330
Noise - Street/Sidewalk,1813
UNSANITARY CONDITION,1737
PLUMBING,1076
Abandoned Vehicle,1050
Dirty Condition,1020


## 9) Convertir a Delta Lake y probar funcionalidades

Aquí construimos una tabla Delta a partir del dataset de taxis filtrado.

In [0]:

delta_taxi_path = f"{silver_path}/taxi_delta"

(
    taxi_clean.write
    .format("delta")
    .mode("overwrite")
    .save(delta_taxi_path)
)

df_delta = spark.read.format("delta").load(delta_taxi_path)
display(df_delta.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,speed_mph
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,8.35,2025-01-01,0,11.497005988023954
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.55,2025-01-01,0,11.764705882352942
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1.95,2025-01-01,0,18.46153846153846
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,5.566666666666666,2025-01-01,0,5.604790419161676
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,3.533333333333333,2025-01-01,0,11.207547169811322
2,2025-01-01T00:48:24.000,2025-01-01T01:08:26.000,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1.0,24.1,2.5,0.0,0.0,20.033333333333335,2025-01-01,0,7.876871880199666
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,1.4666666666666666,2025-01-01,0,16.363636363636367
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,12.4,2025-01-01,0,7.741935483870968
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,19.666666666666668,2025-01-01,0,8.54237288135593
2,2025-01-01T00:00:02.000,2025-01-01T00:09:36.000,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1.0,16.4,2.5,0.0,0.0,9.566666666666666,2025-01-01,0,10.724738675958188


In [0]:
# Registrar como vista temporal para consultas SQL
spark.sql("DROP VIEW IF EXISTS demo_taxi_delta")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW demo_taxi_delta
AS SELECT * FROM delta.`{delta_taxi_path}`
""")

display(spark.sql("SELECT COUNT(*) AS total_rows FROM demo_taxi_delta"))

total_rows
2838026


In [0]:
# Historial de Delta
display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
14,2026-04-11T04:25:24.000Z,6553639838337713,emontoya@eafit.edu.co,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2946777719161701),88c47026-78a9-45c0-b7e9-ffcb877fbd5d,0411-042336-vl716xn4-v2n,13,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 81477411, numDeletionVectorsRemoved -> 0, numOutputRows -> 2838026, numOutputBytes -> 83940922)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-11T03:42:56.000Z,6553639838337713,emontoya@eafit.edu.co,MERGE,"Map(predicate -> [""(cast(VendorID#24709 as bigint) = VendorID#24733L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(2946777719161701),8956b112-f802-4f2b-8204-579b11ff5f22,0411-030538-tmnnei8h-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 81477411, numTargetBytesRemoved -> 81477651, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2838026, executionTimeMs -> 4704, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1717, numTargetRowsUpdated -> 2838026, numOutputRows -> 2838026, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2854)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
12,2026-04-11T03:42:47.000Z,6553639838337713,emontoya@eafit.edu.co,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2946777719161701),40b344f4-e647-4f46-a0c6-e62b5ff18262,0411-030538-tmnnei8h-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 83940922, p25FileSize -> 69939981, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 43, minFileSize -> 69939981, numAddedFiles -> 1, maxFileSize -> 69939981, p75FileSize -> 69939981, p50FileSize -> 69939981, numAddedBytes -> 69939981)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
11,2026-04-11T03:42:45.000Z,6553639838337713,emontoya@eafit.edu.co,DELETE,"Map(predicate -> [""(trip_distance#23940 <= 0.0)""])",null,List(2946777719161701),8422ffdb-c99b-48b5-bd8e-5204cc9d05b2,0411-030538-tmnnei8h-v2n,10,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 369, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 368, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
10,2026-04-11T03:42:43.000Z,6553639838337713,emontoya@eafit.edu.co,UPDATE,"Map(predicate -> [""((VendorID#23285 = 1) AND (fare_amount#23295 < 20.0))""])",null,List(2946777719161701),40b344f4-e647-4f46-a0c6-e62b5ff18262,0411-030538-tmnnei8h-v2n,9,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3100, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1048, numAddedFiles -> 1, numUpdatedRows -> 500794, numAddedBytes -> 11537670, rewriteTimeMs -> 2049)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
9,2026-04-11T03:42:33.000Z,6553639838337713,emontoya@eafit.edu.co,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2946777719161701),6c0d488e-2339-4c95-831c-5458ed6ef267,0411-030538-tmnnei8h-v2n,8,WriteSerializable,false,"Map(numFiles

In [0]:
# UPDATE y DELETE en Delta con SQL
spark.sql(f"""
UPDATE delta.`{delta_taxi_path}`
SET fare_amount = fare_amount * 1.02
WHERE VendorID = 1 AND fare_amount < 20
""")

spark.sql(f"""
DELETE FROM delta.`{delta_taxi_path}`
WHERE trip_distance <= 0
""")

display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
16,2026-04-11T04:25:44.000Z,6553639838337713,emontoya@eafit.edu.co,DELETE,"Map(predicate -> [""(trip_distance#14393 <= 0.0)""])",null,List(2946777719161701),d5bb098c-43f4-4a80-88f4-41cd6f5029da,0411-042336-vl716xn4-v2n,15,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 836, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 792, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
15,2026-04-11T04:25:40.000Z,6553639838337713,emontoya@eafit.edu.co,UPDATE,"Map(predicate -> [""((VendorID#13712 = 1) AND (fare_amount#13722 < 20.0))""])",null,List(2946777719161701),838eba2f-7317-48eb-acd4-fa47de3341a5,0411-042336-vl716xn4-v2n,14,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5477, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2253, numAddedFiles -> 1, numUpdatedRows -> 500794, numAddedBytes -> 11537670, rewriteTimeMs -> 3059)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-11T04:25:24.000Z,6553639838337713,emontoya@eafit.edu.co,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2946777719161701),88c47026-78a9-45c0-b7e9-ffcb877fbd5d,0411-042336-vl716xn4-v2n,13,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 81477411, numDeletionVectorsRemoved -> 0, numOutputRows -> 2838026, numOutputBytes -> 83940922)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-11T03:42:56.000Z,6553639838337713,emontoya@eafit.edu.co,MERGE,"Map(predicate -> [""(cast(VendorID#24709 as bigint) = VendorID#24733L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(2946777719161701),8956b112-f802-4f2b-8204-579b11ff5f22,0411-030538-tmnnei8h-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 81477411, numTargetBytesRemoved -> 81477651, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2838026, executionTimeMs -> 4704, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1717, numTargetRowsUpdated -> 2838026, numOutputRows -> 2838026, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2854)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
12,2026-04-11T03:42:47.000Z,6553639838337713,emontoya@eafit.edu.co,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2946777719161701),40b344f4-e647-4f46-a0c6-e62b5ff18262,0411-030538-tmnnei8h-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 83940922, p25FileSize -> 69939981, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 43, minFileSize -> 69939981, numAddedFiles -> 1, maxFileSize -> 69939981, p75FileSize -> 69939981, p50FileSize -> 69939981, numAddedBytes -> 69939981)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
11,2026-04-11T03:42:45.000Z,6553639838337713,emontoya@eafit.edu.co,DELETE,"Map(predicate -> [""(trip_distance#23940 <= 0.0)""])",null,List(2946777719161701),8422ffdb-c99b-48b5-bd8e-5204cc9d05b2,0411-030538-tmnnei8h-v2n,10,WriteSerializable,false,"Map(numRemovedFiles

In [0]:
# MERGE INTO con datos de ejemplo
updates_df = spark.createDataFrame(
    [
        (1, "promo_applied"),
        (2, "standard")
    ],
    ["VendorID", "pricing_tag"]
)

updates_df.createOrReplaceTempView("vendor_updates")

spark.sql("DROP TABLE IF EXISTS vendor_updates_tbl")
spark.sql("CREATE TABLE vendor_updates_tbl USING DELTA AS SELECT * FROM vendor_updates")

spark.sql(f"""
MERGE INTO delta.`{delta_taxi_path}` AS t
USING vendor_updates_tbl AS s
ON t.VendorID = s.VendorID
WHEN MATCHED THEN
  UPDATE SET t.congestion_surcharge = coalesce(t.congestion_surcharge, 0.0)
""")

display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
18,2026-04-11T04:26:02.000Z,6553639838337713,emontoya@eafit.edu.co,MERGE,"Map(predicate -> [""(cast(VendorID#15163 as bigint) = VendorID#15187L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(2946777719161701),7d61aa47-3ab6-4fd5-ad61-40ae66613b8f,0411-042336-vl716xn4-v2n,17,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 81477411, numTargetBytesRemoved -> 81477651, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2838026, executionTimeMs -> 5618, materializeSourceTimeMs -> 11, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2398, numTargetRowsUpdated -> 2838026, numOutputRows -> 2838026, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3059)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
17,2026-04-11T04:25:48.000Z,6553639838337713,emontoya@eafit.edu.co,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2946777719161701),838eba2f-7317-48eb-acd4-fa47de3341a5,0411-042336-vl716xn4-v2n,15,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 83940922, p25FileSize -> 69939981, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 343, minFileSize -> 69939981, numAddedFiles -> 1, maxFileSize -> 69939981, p75FileSize -> 69939981, p50FileSize -> 69939981, numAddedBytes -> 69939981)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
16,2026-04-11T04:25:44.000Z,6553639838337713,emontoya@eafit.edu.co,DELETE,"Map(predicate -> [""(trip_distance#14393 <= 0.0)""])",null,List(2946777719161701),d5bb098c-43f4-4a80-88f4-41cd6f5029da,0411-042336-vl716xn4-v2n,15,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 836, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 792, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
15,2026-04-11T04:25:40.000Z,6553639838337713,emontoya@eafit.edu.co,UPDATE,"Map(predicate -> [""((VendorID#13712 = 1) AND (fare_amount#13722 < 20.0))""])",null,List(2946777719161701),838eba2f-7317-48eb-acd4-fa47de3341a5,0411-042336-vl716xn4-v2n,14,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5477, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2253, numAddedFiles -> 1, numUpdatedRows -> 500794, numAddedBytes -> 11537670, rewriteTimeMs -> 3059)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
14,2026-04-11T04:25:24.000Z,6553639838337713,emontoya@eafit.edu.co,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(2946777719161701),88c47026-78a9-45c0-b7e9-ffcb877fbd5d,0411-042336-vl716xn4-v2n,13,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 81477411, numDeletionVectorsRemoved -> 0, numOutputRows -> 2838026, numOutputBytes -> 83940922)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-11T03:42:56.000Z,6553639838337713,emontoya@eafit.edu.co,MERGE,"Map(predicate -> [""(cast(VendorID#24709 as bigint) = VendorID#24733L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredic

In [0]:
# Time travel: leer una versión anterior, si existe
history = spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`").collect()
versions = [row["version"] for row in history]

print("Versiones disponibles:", versions)

if len(versions) >= 2:
    older_version = sorted(versions)[-2]
    print("Leyendo versión anterior:", older_version)
    older_df = spark.read.format("delta").option("versionAsOf", older_version).load(delta_taxi_path)
    display(older_df.limit(10))
else:
    print("Aún no hay suficientes versiones para time travel.")

Versiones disponibles: [18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
Leyendo versión anterior: 17


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,speed_mph
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.2,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,8.35,2025-01-01,0,11.497005988023954
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.202,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.55,2025-01-01,0,11.764705882352942
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.202,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1.95,2025-01-01,0,18.46153846153846
1,2025-01-01T00:14:47.000,2025-01-01T00:16:15.000,0,0.4,1,N,170,170,1,4.488,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0,1.4666666666666666,2025-01-01,0,16.363636363636367
1,2025-01-01T00:39:27.000,2025-01-01T00:51:51.000,0,1.6,1,N,234,148,1,12.342,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,0.0,12.4,2025-01-01,0,7.741935483870968
1,2025-01-01T00:53:43.000,2025-01-01T01:13:23.000,0,2.8,1,N,148,170,1,19.482000000000003,3.5,0.5,3.0,0.0,1.0,27.1,2.5,0.0,0.0,19.666666666666668,2025-01-01,0,8.54237288135593
1,2025-01-01T00:30:07.000,2025-01-01T00:36:48.000,1,1.1,1,N,229,141,2,8.058,3.5,0.5,0.0,0.0,1.0,12.9,2.5,0.0,0.0,6.683333333333334,2025-01-01,0,9.875311720698255
1,2025-01-01T00:16:54.000,2025-01-01T00:35:12.000,3,2.5,1,N,158,170,2,18.054,3.5,0.5,0.0,0.0,1.0,22.7,2.5,0.0,0.0,18.3,2025-01-01,0,8.19672131147541
1,2025-01-01T00:43:10.000,2025-01-01T01:00:03.000,1,1.9,1,N,164,229,1,16.626,3.5,0.5,4.25,0.0,1.0,25.55,2.5,0.0,0.0,16.883333333333333,2025-01-01,0,6.7522211253701885
1,2025-01-01T00:33:12.000,2025-01-01T00:50:14.000,2,1.2,1,N,246,90,1,15.911999999999999,3.5,0.5,0.0,0.0,1.0,20.6,2.5,0.0,0.0,17.033333333333335,2025-01-01,0,4.227005870841487


# Time travel: leer una versión anterior, si existe
history = spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`").collect()
versions = [row["version"] for row in history]

print("Versiones disponibles:", versions)

if len(versions) >= 2:
    older_version = sorted(versions)[-2]
    print("Leyendo versión anterior:", older_version)
    older_df = spark.read.format("delta").option("versionAsOf", older_version).load(delta_taxi_path)
    display(older_df.limit(10))
else:
    print("Aún no hay suficientes versiones para time travel.")

In [0]:

# Bronze: 311 crudo en Delta
bronze_311_path = f"{bronze_path}/311_delta"

(
    df_csv.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_311_path)
)

# Silver: normalización mínima
df_311_silver = (
    spark.read.format("delta").load(bronze_311_path)
    .withColumn("created_date_ts", F.to_timestamp("created_date"))
    .withColumn("closed_date_ts", F.to_timestamp("closed_date"))
    .withColumn("resolution_days",
                (F.col("closed_date_ts").cast("long") - F.col("created_date_ts").cast("long")) / 86400.0)
)

silver_311_path = f"{silver_path}/311_delta"
df_311_silver.write.format("delta").mode("overwrite").save(silver_311_path)

# Gold: agregado por borough y complaint_type
df_311_gold = (
    spark.read.format("delta").load(silver_311_path)
    .groupBy("borough", "complaint_type")
    .agg(F.count("*").alias("num_requests"),
         F.avg("resolution_days").alias("avg_resolution_days"))
)

gold_311_path = f"{gold_path}/311_agg_delta"
df_311_gold.write.format("delta").mode("overwrite").save(gold_311_path)

display(spark.read.format("delta").load(gold_311_path).orderBy(F.desc("num_requests")).limit(20))

borough,complaint_type,num_requests,avg_resolution_days
BROOKLYN,Illegal Parking,3099,0.10153362728528362
QUEENS,Illegal Parking,2570,0.11044504593175859
BROOKLYN,Noise - Residential,1752,0.09820991694137421
BRONX,Noise - Residential,1633,0.12285211420719395
BRONX,HEAT/HOT WATER,1599,1.266542548844275
BROOKLYN,HEAT/HOT WATER,1320,1.217176411551413
BRONX,Illegal Parking,1264,0.11264461992234168
QUEENS,Noise - Residential,1139,0.11229573575817975
QUEENS,Street Condition,1119,0.43557797729738085
MANHATTAN,HEAT/HOT WATER,1069,1.0154001169655114


## 11) Ejemplos batch avanzados: ventanas, rankings, deduplicación

In [0]:

from pyspark.sql.window import Window

# Ranking de zonas por recaudo dentro de cada vendor
zone_revenue = (
    taxi_clean.groupBy("VendorID", "PULocationID")
    .agg(F.sum("total_amount").alias("revenue"))
)

w = Window.partitionBy("VendorID").orderBy(F.desc("revenue"))

zone_ranked = (
    zone_revenue
    .withColumn("rank_in_vendor", F.row_number().over(w))
)

display(zone_ranked.filter(F.col("rank_in_vendor") <= 10).orderBy("VendorID", "rank_in_vendor"))

VendorID,PULocationID,revenue,rank_in_vendor
1,138,2000381.7499999637,1
1,132,1462031.1699998733,2
1,161,751464.4600000092,3
1,236,679765.2299999926,4
1,237,678895.7199999995,5
1,230,573548.9800000073,6
1,186,572637.5500000056,7
1,162,546689.9700000063,8
1,142,475474.7099999998,9
1,163,439431.9300000034,10


In [0]:

# Detección y eliminación de duplicados en 311 (ejemplo batch)
dedup_keys = ["unique_key"]

df_311_dedup = (
    spark.read.format("delta").load(bronze_311_path)
    .dropDuplicates(dedup_keys)
)

print("Bronze filas:", spark.read.format("delta").load(bronze_311_path).count())
print("Sin duplicados:", df_311_dedup.count())

Bronze filas: 50000
Sin duplicados: 50000


In [0]:
# Cache / persist para acelerar reutilización en análisis batch
# Nota: .cache() no está soportado en serverless compute
print("Filas:", taxi_clean.count())

Filas: 2838026


## 12) ML supervisado: regresión lineal

Ejemplo sencillo: predecir `tip_amount` a partir de variables operativas del viaje.

In [0]:

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

lr_df = (
    taxi_clean
    .select(
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "trip_duration_min",
        "tip_amount"
    )
    .dropna()
    .filter(F.col("tip_amount") >= 0)
)

train_df, test_df = lr_df.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(
    inputCols=["passenger_count", "trip_distance", "fare_amount", "trip_duration_min"],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="tip_amount",
    predictionCol="prediction",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[assembler, lr])
lr_model = pipeline.fit(train_df)

pred_lr = lr_model.transform(test_df)

display(pred_lr.select("tip_amount", "prediction").limit(20))

evaluator_rmse = RegressionEvaluator(labelCol="tip_amount", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="tip_amount", predictionCol="prediction", metricName="r2")

print("RMSE:", evaluator_rmse.evaluate(pred_lr))
print("R2:", evaluator_r2.evaluate(pred_lr))

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-4914540249399454>, line 21
      6 lr_df = (
      7     taxi_clean
      8     .select(
   (...)
     16     .filter(F.col("tip_amount") >= 0)
     17 )
     19 train_df, test_df = lr_df.randomSplit([0.8, 0.2], seed=42)
---> 21 assembler = VectorAssembler(
     22     inputCols=["passenger_count", "trip_distance", "fare_amount", "trip_duration_min"],
     23     outputCol="features"
     24 )
     26 lr = LinearRegression(
     27     featuresCol="features",
     28     labelCol="tip_amount",
   (...)
     32     elasticNetParam=0.0
     33 )
     35 pipeline = Pipeline(stages=[assembler, lr])

File /databricks/python/lib/python3.11/site-packages/pyspark/__init__.py:120, in keyword_only.<locals>.wrapper(self, *args, **kwargs)
    118     raise TypeError("Method %s forces keyword arguments." % func.__name__)
    119 self._

## 13) K-Means

Clustering sencillo de viajes según distancia, tarifa y duración.

In [0]:

from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StandardScaler

km_df = (
    taxi_clean
    .select("trip_distance", "fare_amount", "trip_duration_min")
    .dropna()
)

train_km, test_km = km_df.randomSplit([0.8, 0.2], seed=42)

assembler_km = VectorAssembler(
    inputCols=["trip_distance", "fare_amount", "trip_duration_min"],
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

kmeans = KMeans(
    k=4,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

km_pipeline = Pipeline(stages=[assembler_km, scaler, kmeans])
km_model = km_pipeline.fit(train_km)

clustered = km_model.transform(test_km)
display(clustered.limit(20))

In [0]:

# Tamaños de cluster
display(
    clustered.groupBy("cluster")
    .agg(
        F.count("*").alias("n"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_duration_min").alias("avg_duration")
    )
    .orderBy("cluster")
)

## 14) SQL directo sobre Delta

In [0]:

spark.sql("DROP VIEW IF EXISTS gold_requests")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW gold_requests
AS SELECT * FROM delta.`{gold_311_path}`
""")

display(spark.sql("""
SELECT borough, SUM(num_requests) AS total_requests
FROM gold_requests
GROUP BY borough
ORDER BY total_requests DESC
"""))

## 15) Opcionales Databricks-only

Estas celdas son útiles si su workspace permite comandos de mantenimiento como `OPTIMIZE` y `VACUUM`.
Descomente si quiere usarlas.

In [0]:

# spark.sql("OPTIMIZE demo_taxi_delta ZORDER BY (VendorID, PULocationID)")
# spark.sql("VACUUM demo_taxi_delta RETAIN 168 HOURS")

## 16) Limpieza opcional

In [0]:

# Descomente si desea borrar todo el laboratorio
# dbutils.fs.rm(base_path, True)